# Data Exploration

Programmatically validate the 8 confirmed bug counts before pipeline execution.

In [ ]:
import pandas as pd
import duckdb
import json
from decimal import Decimal
from pathlib import Path

data_dir = Path("data")

logs = pd.read_csv(data_dir / "supervisor_logs.csv", dtype=str)
transfers = pd.read_csv(data_dir / "bank_transfers.csv", dtype=str)
rates = pd.read_csv(data_dir / "wage_rates.csv", dtype=str)
workers = pd.read_csv(data_dir / "workers.csv", dtype=str)

### 1. Suspected Unit Error

In [ ]:
transfers["amount_paise"] = transfers["amount_paise"].apply(lambda x: Decimal(str(x)))
unit_errors = transfers[transfers["amount_paise"] < 5000]
print(f"Suspected unit errors: {len(unit_errors)}")

### 2. Rate Overlap

In [ ]:
con = duckdb.connect()
con.register("shifts", logs)
con.register("wage_rates", rates)
con.register("workers", workers)

res = con.execute("""
WITH joined AS (
    SELECT s.log_id, COUNT(*) as match_count
    FROM shifts s
    JOIN workers w ON w.phone LIKE "%" || SUBSTR(REPLACE(REPLACE(REPLACE(s.worker_phone, ' ', ''), '-', ''), '+', ''), -10, 10)
    JOIN wage_rates r ON r.role = w.role AND r.state = w.state AND r.seniority = w.seniority
    WHERE r.effective_from <= CAST(s.work_date AS DATE)
    AND (r.effective_to IS NULL OR r.effective_to = '' OR TRY_CAST(r.effective_to AS DATE) >= CAST(s.work_date AS DATE))
    GROUP BY s.log_id
)
SELECT COUNT(*) FROM joined WHERE match_count > 1
""").fetchone()[0]
print(f"Rate overlaps: {res}")
con.close()

### 3. Timezone Boundary Risk

In [ ]:
logs["entered_at_ist"] = pd.to_datetime(logs["entered_at"], utc=True).dt.tz_convert("Asia/Kolkata")
# Simplified count for vendor_b_v1.0
tz_risk = len(logs[(logs["vendor_app"] == "vendor_b_v1.0") & (logs["entered_at_ist"].dt.date.astype(str) != logs["work_date"])])
print(f"Timezone boundary risks: {tz_risk}")

### 4. Invalid Hours

In [ ]:
logs["hours_num"] = pd.to_numeric(logs["hours"], errors="coerce")
invalid_hours = len(logs[(logs["hours_num"] <= 0) | (logs["hours_num"] > 24)])
print(f"Invalid hours: {invalid_hours}")

### 5. Backdated Crosses Cycle

In [ ]:
logs["work_date_dt"] = pd.to_datetime(logs["work_date"])
logs["lag"] = (logs["entered_at_ist"].dt.date - logs["work_date_dt"].dt.date).dt.days
backdated = len(logs[(logs["lag"] > 7) & (logs["entered_at_ist"].dt.month != logs["work_date_dt"].dt.month)])
print(f"Backdated crosses cycle: {backdated}")

### 8. Rate Precision Anomaly

In [ ]:
rates["hourly_rate_inr"] = rates["hourly_rate_inr"].apply(lambda x: Decimal(str(x)))
anomaly = rates[rates["hourly_rate_inr"] != rates["hourly_rate_inr"].apply(lambda x: x.to_integral_value())]
print(f"Rate precision anomalies in wage_rates: {len(anomaly)}")